# Behavior Modeling API: SMART Implementation

## Introduction

Tactics2D provides unified reimplementations of a collection of representative traffic participant behavior models to support the development, validation, and testing of Autonomous Driving Systems (ADS) with realistic and scalable traffic interactions. SMART is one of the default behavior models integrated into Tactics2D.

Original paper: [SMART: Scalable Multi-agent Real-time Motion Generation via Next-token Prediction](https://arxiv.org/abs/2405.15677)

Original code: [rainmaker22/SMART](https://github.com/rainmaker22/SMART)

SMART tokenizes every agent's motion history and the map against two learned codebooks, then **decodes the scene jointly**: one encoder pass produces the future of every modelled agent at once, and the rollout is an autoregressive next-token prediction over the motion codebook. Tactics2D ports inference - the tokenizers, the encoders, the decoder and the closed-loop runner - not training.


## Environment Setup

Please install Tactics2D (`pip install 'tactics2d[behavior]'`) or add the Tactics2D source directory to your `PYTHONPATH`. See the [Installation Guide](https://tactics2d.readthedocs.io/en/latest/installation/) for more details. SMART additionally requires `torch`.

The four behavior demos share their scaffolding: scenario selection, map loading, replay rendering and error scoring. It lives in [`tutorial_common.py`](../tutorial_common/), imported below as `tutorial_common`; each notebook only adds what is specific to its model.

## Model Preparation

The checkpoint and the two codebooks are hosted on [Hugging Face](https://huggingface.co/MotacillaAlba/tactics2d-behavior) and are **not redistributed with Tactics2D**. All three belong in one directory, which `SmartConfig(asset_root=...)` points at.

| File | Role |
|------|------|
| `smart_waymo_ep0.pt` | the released SMART encoder weights, exported from the official Lightning checkpoint |
| `motion_codebook.pkl` | 2048-entry motion codebook, used to tokenize agent histories |
| `map_codebook.pkl` | 1024-entry map codebook, used to tokenize map polylines |

Download them with `hf download`, run from the repository root:

```bash
hf download MotacillaAlba/tactics2d-behavior --include "smart/*" --local-dir checkpoints
```

!!! warning "There is no default codebook path"
    A bare `SmartConfig()` resolves no codebook at all: loading one raises a `ValueError` naming the missing asset rather than falling back to a copy bundled inside the package. `asset_root` is what expands into `<asset_root>/motion_codebook.pkl` and `<asset_root>/map_codebook.pkl`; the two paths can also be passed separately.

## Dataset Preparation

Tactics2D does not require datasets to be stored in a fixed location. You can place a dataset in any directory and provide its path when parsing it. Adjust the paths below to match your local data layout.

| Dataset | Role here | Rate |
|---------|-----------|------|
| **WOMD** | the model's own dataset: SMART is trained on it, and its tokenizer reads WOMD's lane, road-line and traffic-light vocabulary directly | 10 Hz |
| **nuPlan** | off-domain: a different city, a different collection, and no shared vocabulary | 20 Hz |

!!! warning "The model runs on a fixed 100 ms lattice"
    Every behavior model lays a scenario out on a fixed step - SMART's is `step_ms = 100`, and its history window and pose arrays are indexed on it. A log recorded at another rate is **resampled onto that lattice automatically** by the runner (`tactics2d.behavior.rolling_utils.to_lattice`), interpolating positions and headings; feeding it as-is would land several frames on one index and leave the history window full of holes.

    Note that no other 10 Hz dataset is set up here, so every off-domain example below is also an off-rate one. That is a gap in the data on hand, not a step the demos skipped.


## Use SMART for Behavior Generation

Both usages below go through the same public API; the notebook only provides glue code.

| Module | Key API |
|--------|---------|
| **Dataset parsers** | `parser.parse_trajectory(...)` -> `(participants, time_range)`; `parser.parse_map(...)` -> `Map` |
| **Behavior model** | `SmartBehaviorModel.from_checkpoint(...)` -> `.predict(...)`, `.rollout(...)` |
| **Resampling** | `tactics2d.behavior.rolling_utils.to_lattice(participants, step_ms)` |
| **Rendering** | `BEVCamera` + `MatplotlibRenderer`, driven through `tutorial_common.render_replay_animation` |


In [1]:
import warnings

warnings.filterwarnings("ignore")

import logging

# WOMD's timestamps drift, so per-trajectory warnings would bury the demos' output.
logging.basicConfig(level=logging.ERROR)


import tutorial_common
from tactics2d.behavior import SmartBehaviorModel, SmartConfig
from tactics2d.behavior.rolling_utils import to_lattice
from tactics2d.dataset_parser import LevelXParser, NuPlanParser, WOMDParser
from tactics2d.map.map_config import IND_MAP_CONFIG

pygame 2.6.1 (SDL 2.28.4, Python 3.10.20)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
tutorial_common.apply_notebook_style()

In [3]:
# ---- Model assets ----
ASSET_ROOT = "../../checkpoints/smart"
CHECKPOINT = f"{ASSET_ROOT}/smart_waymo_ep0.pt"
DEVICE = "cpu"  # "cuda" is ~6x faster per rollout when a GPU is free

# ---- Dataset ----
# WOMD shards are named "<split>.tfrecord-000XX-of-00150"; the demos read shard 0.
WOMD_ROOT = "../../../data/womd/uncompressed"

# The rollout samples its next token; a pinned seed makes it reproducible.
config = SmartConfig(asset_root=ASSET_ROOT, seed=0)
model = SmartBehaviorModel.from_checkpoint(CHECKPOINT, config=config, device=DEVICE)


def womd_paths(split, shard=0):
    return f"{split}.tfrecord-{shard:05d}-of-00150", f"{WOMD_ROOT}/{split}"


print("history_steps:", config.history_steps, "  future_steps:", config.future_steps)
print("dt:", config.dt, "s  (step_ms:", config.step_ms, "ms)")
print("beam_size:", config.beam_size, "  seed:", config.seed)
print("predicted_radius:", config.predicted_radius, "m")
print("max_predicted_agents:", config.max_predicted_agents)
print("modelled classes:", "vehicles only" if config.vehicle_only else "every class")

history_steps: 11   future_steps: 80
dt: 0.1 s  (step_ms: 100 ms)
beam_size: 5   seed: 0
predicted_radius: 100.0 m
max_predicted_agents: 32
modelled classes: vehicles only


## Take-Over Usage

`predict()` replaces the future of **one** vehicle and leaves everybody else on the trajectory the log recorded. It is the call all four Tactics2D behavior models share, and the one the cross-model comparison is run on.

```python
predicted = model.predict(participants, map_, frame=TAKEOVER_FRAME, agent_ids=[ego_id])
```

`frame` is the newest frame the model conditions on, in milliseconds, and the return value is `{agent_id: Trajectory}`. Even when a single id is asked for the scene is still decoded jointly - the neighbours' committed tokens are part of every agent's input - and the returned dict is only filtered afterwards. The prediction is scored against the vehicle's recorded future with `tutorial_common.displacement_errors`.


### Step 1: Select the Vehicle

`Vehicle` carries a `Trajectory` (a `dict[int, State]` mapping timestamps to positions) and a `color` attribute the renderer reads. `tutorial_common.select_ego` picks a vehicle that is present before the warm-up frame and survives into the second half of the scenario; passing an explicit `ego_id` is just as valid when a particular vehicle is the one under study.


### Step 2: Predict, and Score Against the Recorded Future

`SmartBehaviorModel.predict(...)` takes the parsed `participants` and `map_` directly. The trigger frame below is the shared one - the same junction, the same vehicle and the same frame the LimSim and InterSim demos use - so the numbers can be read side by side.

SMART stamps the predicted frames on **the scenario's own grid**: its rollout runs at the same 10 Hz the log was recorded at, so the states can be written straight onto the vehicle's trajectory, frame for frame, with no interpolation.


### Step 3: Render with BEVCamera and MatplotlibRenderer

The animation comes from `tutorial_common.render_replay_animation`. The dashed history is what the log recorded, the solid trace is the prediction, and the purple gradient behind the vehicle is the path it has already driven. Everything outside the modelled set keeps its recorded motion, so the frame is directly comparable to the closed-loop replay further down.


In [4]:
parser = WOMDParser()
file_name, folder = (
    tutorial_common.COMPARISON_FILE,
    f"{WOMD_ROOT}/{tutorial_common.COMPARISON_SPLIT}",
)
participants, time_range = parser.parse_trajectory(
    tutorial_common.COMPARISON_SCENARIO, file=file_name, folder=folder
)
participants = to_lattice(participants, model.config.step_ms)
map_ = parser.parse_map(tutorial_common.COMPARISON_SCENARIO, file=file_name, folder=folder)

# The same junction, vehicle and take-over frame as the LimSim and InterSim demos.
ego_id = tutorial_common.COMPARISON_EGO
TAKEOVER_FRAME = tutorial_common.COMPARISON_FRAME_MS
truth = tutorial_common.recorded_future(participants[ego_id].trajectory, TAKEOVER_FRAME)

predicted = model.predict(participants, map_, frame=TAKEOVER_FRAME, agent_ids=[ego_id])
plan = predicted[ego_id]
ade, fde, matched = tutorial_common.displacement_errors(
    plan, truth, tutorial_common.COMPARISON_HORIZON_STEPS
)
own_ade, own_fde, own_matched = tutorial_common.displacement_errors(plan, truth)
print(
    f"ego: {ego_id}  |  predicted {len(plan.frames)} steps, "
    f"{min(plan.frames)}-{max(plan.frames)} ms"
)
print(f"  vs the recorded future:  ADE@2s {ade:.3f} m  FDE@2s {fde:.3f} m  ({matched} steps)")
print(
    f"                           ADE@8s {own_ade:.3f} m  FDE@8s {own_fde:.3f} m  "
    f"({own_matched} steps)"
)

# Drop the ego's own future, then lay the rollout onto the frames it returned.
ego = participants[ego_id]
ego.color = tutorial_common.EGO_COLOR
ego.trajectory._history_states = {
    frame: state
    for frame, state in ego.trajectory.history_states.items()
    if frame <= TAKEOVER_FRAME
}
ego.trajectory._frames = sorted(ego.trajectory._history_states)
for frame in plan.frames:
    ego.trajectory.add_state(plan.get_state(frame))

playback_frames = [f for f in ego.trajectory.frames if f <= time_range[1]]
ani_takeover = tutorial_common.render_replay_animation(
    participants,
    map_,
    playback_frames,
    ego_id,
    plans={TAKEOVER_FRAME: [(f, plan.get_state(f).x, plan.get_state(f).y) for f in plan.frames]},
    fps=1000.0 / model.config.step_ms,
    title_prefix="SMART take-over",
)
ani_takeover

ego: 8  |  predicted 80 steps, 1200-9100 ms
  vs the recorded future:  ADE@2s 2.479 m  FDE@2s 6.717 m  (20 steps)
                           ADE@8s 31.316 m  FDE@8s 76.504 m  (79 steps)


## Closed-Loop Usage

`rollout()` is the other half: instead of one vehicle, the runner replays the whole scenario. The ego and the vehicles around it are decoded together and committed together, replanning once per second, while every vehicle outside the modelled set keeps its recorded trajectory.

It returns metrics and one pose array per participant rather than an animation, so the last two steps below turn those arrays back into something the camera can draw.


### Step 1: Select the Ego Vehicle

The replay is centred on one vehicle: it drives the loop, and it is the participant the collision and progress metrics are reported for.


### Step 2: Replay the Scenario Closed-Loop

```python
rolling = model.rollout(
    participants, map_, ego_id=ego_id, frame_ms0=0, planning_interval=10
)
```

- `ego_id` - the agent the replay is centred on.
- `warmup_steps` - steps of recorded motion before the first replan. Defaults to the model's history length, 11 frames.
- `planning_interval` - steps between replans. The default 10 is one replan per second at 10 Hz.
- `scenario_steps` - how many steps to replay. Defaults to the model's whole token span, `history_steps + future_steps = 91` (9.1 s).

The result is a `SmartRollingResult`: `collided` plus a `front` / `side` / `rear` breakdown, `progress` in metres, `total_agents_controlled`, `modelled_ids`, and `poses` - one `(steps, 4)` array of `[x, y, z, yaw]` per participant, with `-1` where that participant is absent. The replay stops at the first collision involving the ego.


### Step 3: Render with BEVCamera and MatplotlibRenderer

Two helpers in `tutorial_common` bridge the runner's pose arrays to `render_replay_animation`:

- `tutorial_common.rebuild_render_participants` turns each pose array back into a `Trajectory` and a participant object, mirroring the runner's own snapshot so the rebuilt scene is the one the model replanned on.
- `tutorial_common.collect_ego_plans` recovers the ego's plan at each planning frame; the runner commits its plans into the pose arrays but never returns them, so the plan trace is re-derived by asking the same model, at the same frames, on the simulated state. The pinned `seed` is what makes that replay land on the committed trajectory rather than near it.

In [5]:
def rollout_scenario(
    parser,
    file_name,
    folder,
    scenario_id=None,
    map_path=None,
    map_config=None,
    ego_id=None,
    frame_ms0=None,
    planning_interval=10,
    scenario_steps=None,
    resolution=(1200, 800),
    **parse_kwargs,
):
    print("Parsing scenario ...")
    participants, time_range = parser.parse_trajectory(
        *([scenario_id] if scenario_id is not None else []),
        file=file_name,
        folder=folder,
        **parse_kwargs,
    )
    participants = to_lattice(participants, model.config.step_ms)

    map_ = tutorial_common.load_map(
        parser,
        file_name=file_name,
        folder=folder,
        map_path=map_path,
        map_config=map_config,
        **({"scenario_id": scenario_id} if scenario_id is not None else {}),
        **parse_kwargs,
    )
    print(f"  participants: {len(participants)},  frames: {time_range},  lanes: {len(map_.lanes)}")

    if ego_id is None:
        ego_id = tutorial_common.select_ego(participants)
    if frame_ms0 is None:
        frame_ms0 = int(participants[ego_id].trajectory.first_frame)
    ego = participants[ego_id]
    ego.color = tutorial_common.EGO_COLOR
    print(f"  ego: {ego_id}  (track {ego.trajectory.first_frame}-{ego.trajectory.last_frame} ms)")

    rolling = model.rollout(
        participants,
        map_,
        ego_id=ego_id,
        frame_ms0=frame_ms0,
        planning_interval=planning_interval,
        scenario_steps=scenario_steps,
    )
    print(
        "  closed loop: collided=%s  front/side/rear=%d/%d/%d  progress=%.2f m  "
        "controlled=%d"
        % (
            rolling.collided,
            rolling.front_collisions,
            rolling.side_collisions,
            rolling.rear_collisions,
            rolling.progress,
            rolling.total_agents_controlled,
        )
    )

    step_ms = model.config.step_ms
    render_participants = tutorial_common.rebuild_render_participants(
        rolling.poses, participants, step_ms, frame_ms0
    )
    render_participants[ego_id].color = tutorial_common.EGO_COLOR
    steps = next(iter(rolling.poses.values())).shape[0]
    playback_frames = [frame_ms0 + index * step_ms for index in range(steps)]

    def plan_at(frame):
        return model.predict_scene(render_participants, map_, frame, agent_ids=[ego_id]).trajectory(
            ego_id
        )

    plans = tutorial_common.collect_ego_plans(
        plan_at, model.config.history_steps, steps, planning_interval, step_ms, frame_ms0
    )
    print(f"  rendering {len(playback_frames)} frames, {len(plans)} re-derived ego plans ...")

    return tutorial_common.render_replay_animation(
        render_participants,
        map_,
        playback_frames,
        ego_id,
        plans=plans,
        resolution=resolution,
        fps=1000.0 / step_ms,
        title_prefix="SMART closed loop",
    )

### Example 1: WOMD - validation_interactive, Scenario 2 (the model's own dataset)

`validation_interactive` collects WOMD scenarios in which the ego actually has to negotiate with someone - and this particular junction is the scene every behavior demo is measured on, so LimSim, InterSim and SMART all replay the same vehicle `8` here.

34 participants, 213 lanes, vehicles crossing the ego's path in both directions, and 31 of them inside the model's 100 m modelling radius. The plan trace is redrawn at each of the eight replans: watch it extend ahead of the ego and shorten again as the vehicle drives into it.


In [6]:
file_name, folder = womd_paths(tutorial_common.COMPARISON_SPLIT)
ani_vi2 = rollout_scenario(
    WOMDParser(),
    file_name,
    folder,
    tutorial_common.COMPARISON_SCENARIO,
    ego_id=tutorial_common.COMPARISON_EGO,  # tutorial_common.select_ego(participants) picks a stationary vehicle in this scenario
)
ani_vi2

Parsing scenario ...
  participants: 34,  frames: (0, 8976),  lanes: 213
  ego: 8  (track 0-8976 ms)
  closed loop: collided=False  front/side/rear=0/0/0  progress=138.62 m  controlled=31
  rendering 91 frames, 8 re-derived ego plans ...


### Example 2: WOMD - validation_interactive, Scenario 9 (the yielding case)

The same shard, and this time the ego is the one that has to give way: it opens almost stationary, and the relation graph puts it on the reactor side of `2608 -> 2478`. 42 participants and 311 lanes - the busiest scene the demo replays.


In [7]:
file_name, folder = womd_paths(tutorial_common.COMPARISON_SPLIT)
ani_vi9 = rollout_scenario(
    WOMDParser(),
    file_name,
    folder,
    tutorial_common.SECOND_SCENARIO,
    ego_id=tutorial_common.SECOND_EGO,
)
ani_vi9

Parsing scenario ...
  participants: 42,  frames: (0, 9000),  lanes: 311
  ego: 2478  (track 0-9000 ms)
  closed loop: collided=False  front/side/rear=0/0/0  progress=767.77 m  controlled=37
  rendering 91 frames, 8 re-derived ego plans ...


### Example 3: nuPlan - Boston Intersection (off-domain, resampled)

nuPlan sits at the other end of the map scale from WOMD: one log carries a whole city's map, and the scenario is a window cut out of it. The window below is the longest pass through an intersection in the log, recomputed from the log's `scenario_tag` table at run time rather than stored, because the parser stamps frames relative to `datetime(2021, 1, 1)` **in the local timezone** - a hard-coded window would mean a different stretch of road on another machine.

Two things change at once here, and it is worth keeping them apart: the domain is new (different city, different traffic) and the rate is new (20 Hz, resampled onto the model's lattice). The port's cost is dominated by the map rather than the traffic - the map encoder connects every point token to the ones within 10 m, so this city map costs an order of magnitude more than a WOMD scenario map, which is why the cell is slow on CPU and why its plan trace is left off.


In [8]:
# ---- nuPlan, same four models, a city-scale map ----
NUPLAN_ROOT = "../../../data/nuplan"
nuplan_folder = f"{NUPLAN_ROOT}/data/cache/{tutorial_common.NUPLAN_SCENARIO_FOLDER}"

nuplan_parser = NuPlanParser()
nuplan_window = tutorial_common.nuplan_intersection_window(
    f"{nuplan_folder}/{tutorial_common.NUPLAN_SCENARIO_FILE}"
)
nuplan_participants, _ = nuplan_parser.parse_trajectory(
    file=tutorial_common.NUPLAN_SCENARIO_FILE, folder=nuplan_folder, time_range=nuplan_window
)
nuplan_participants = to_lattice(nuplan_participants, model.config.step_ms)
nuplan_map = nuplan_parser.parse_map(
    file="map.gpkg", folder=f"{NUPLAN_ROOT}/maps/{tutorial_common.NUPLAN_SCENARIO_MAP}"
)
# The runner lays the log onto the model's 100 ms lattice itself (see
# tactics2d.behavior.rolling_utils.to_lattice).

nuplan_ego = tutorial_common.NUPLAN_SCENARIO_EGO
nuplan_origin = min(p.trajectory.first_frame for p in nuplan_participants.values())
nuplan_participants[nuplan_ego].color = tutorial_common.EGO_COLOR
print(
    f"window {nuplan_window[0]}-{nuplan_window[1]} ms  |  {len(nuplan_participants)} participants  "
    f"|  {len(nuplan_map.lanes)} lanes"
)

rolling_nuplan = model.rollout(
    nuplan_participants,
    nuplan_map,
    ego_id=nuplan_ego,
    frame_ms0=nuplan_origin,
    warmup_steps=11,
    planning_interval=10,
    scenario_steps=41,
)
print(
    "  closed loop: collided=%s  front/side/rear=%d/%d/%d  progress=%.2f m  controlled=%d"
    % (
        rolling_nuplan.collided,
        rolling_nuplan.front_collisions,
        rolling_nuplan.side_collisions,
        rolling_nuplan.rear_collisions,
        rolling_nuplan.progress,
        rolling_nuplan.total_agents_controlled,
    )
)

step_ms = model.config.step_ms
render_participants = tutorial_common.rebuild_render_participants(
    rolling_nuplan.poses, nuplan_participants, step_ms, nuplan_origin
)
render_participants[nuplan_ego].color = tutorial_common.EGO_COLOR
playback_frames = [
    nuplan_origin + index * step_ms
    for index in range(next(iter(rolling_nuplan.poses.values())).shape[0])
]
ani_nuplan = tutorial_common.render_replay_animation(
    render_participants,
    nuplan_map,
    playback_frames,
    nuplan_ego,
    resolution=(1200, 800),
    fps=1000.0 / step_ms,
    title_prefix="SMART closed loop (nuPlan)",
)
ani_nuplan

window 20572567849-20572584249 ms  |  100 participants  |  3019 lanes
  closed loop: collided=False  front/side/rear=0/0/0  progress=60.25 m  controlled=33


### Example 4: inD - Location 1, Recording 07 (off-domain, resampled)

A German urban junction at 25 Hz, resampled onto the lattice. This is the scene the LimSim demo also uses, so the two sketches of it can be put side by side.


In [9]:
ani_ind = rollout_scenario(
    LevelXParser("inD"),
    file_name=7,
    folder="../../data/inD/data",
    map_path="../../data/inD_map/inD_1.osm",
    map_config=IND_MAP_CONFIG["inD_1"],
    ego_id=12,
)
ani_ind

Parsing scenario ...
  loading map from ../../data/inD_map/inD_1.osm
  participants: 212,  frames: (np.int64(0), np.int64(1055240)),  lanes: 137
  ego: 12  (track 10300-16600 ms)
  closed loop: collided=False  front/side/rear=0/0/0  progress=169.21 m  controlled=9
  rendering 91 frames, 8 re-derived ego plans ...


## Quick Configurations

| Use Case | Key Settings |
|----------|-------------|
| **Fast preview** | `beam_size=1`, `scenario_steps=41` - one greedy token per step, over less than half the model's span |
| **Demo defaults** | `beam_size=5`, `seed=0`, `scenario_steps=None` (the full 91 steps), `planning_interval=10` |
| **Larger neighbourhood** | `predicted_radius` (default 100 m) and `max_predicted_agents` (default 32) size the jointly decoded set |
| **Faster on a GPU** | `device="cuda"` in `from_checkpoint` - the port never forces a device |

!!! warning "Pin the seed when you keep the numbers"
    Each next token is sampled with `torch.multinomial` from the top `beam_size` candidates, so `seed=None` - the default - gives a different rollout on every call. Two replays of the same scenario are not comparable until the seed is pinned, and neither are the committed notebook outputs.
